## بخش ۱ — واردسازی و آماده‌سازی محیط

**هدف این سلول:** واردکردن کتابخانه‌های موردنیاز، تعیین مسیر فایل داده و پوشه پروژه، و ساخت کلاینت OpenAI.

**متغیرها و توابع:** DATA_PATH مسیر فایل اکسل، PROJECT_DIR مسیر خروجی، SEED مقدار ثابت تصادفی و client کلاینت API است. اگر OPENAI_API_KEY در محیط تعریف نشده باشد، کلید به‌صورت مخفی درخواست می‌شود.

**ارتباط با روش RecMind:** این بخش زیرساخت حافظه کاربر، استدلال ترجیح و ارزیابی عامل را آماده می‌کند.

In [ ]:
import os, json, math, random, re
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
PROJECT_DIR = Path.cwd().resolve()
DATA_PATH = Path(os.getenv("DIGIKALA_DATA_PATH", str(PROJECT_DIR / "data" / "keifiat.xlsx")))
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("Set OPENAI_API_KEY in your environment before running API cells.")
client = OpenAI(api_key=api_key, base_url=os.getenv("OPENAI_BASE_URL", "https://api.gapgpt.app/v1"), timeout=60.0)

def gapgpt_chat_completion(messages, **kwargs):
    model_name = kwargs.pop("model", MODEL)
    return client.chat.completions.create(model=model_name, messages=messages, **kwargs)

print("Configuration ready; dataset path:", DATA_PATH)


## بخش ۲ — بارگذاری و بررسی داده

**هدف این سلول:** بارگذاری فایل keifiat.xlsx و مشاهده ابعاد، نام ستون‌ها و چند نمونه متن برای بررسی سالم‌بودن فارسی.

**متغیرها و توابع:** df_raw داده خام خوانده‌شده با pandas است.

**ارتباط با روش RecMind:** بررسی طرح داده مشخص می‌کند تعامل‌ها و متن مرورها از کدام ستون‌ها استخراج می‌شوند.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"فایل دیتاست یافت نشد: {DATA_PATH}")

df_raw = pd.read_excel(DATA_PATH)
print("ابعاد دیتاست:", df_raw.shape)
print("ستون‌ها:", df_raw.columns.tolist())
display(df_raw.head(3))

# پیش‌نمایش چند ستون متنی برای بررسی کیفیت داده
for c in df_raw.select_dtypes(include="object").columns[:5]:
    v = df_raw[c].dropna()
    if len(v):
        print(f"--- نمونه ستون {c} ---")
        print(str(v.iloc[0])[:200])
        print()


## بخش ۳ — پاک‌سازی و یکسان‌سازی داده

**هدف این سلول:** نگه‌داشتن فقط برچسب‌های recommended و not_recommended، پاک‌کردن نویسه‌های اضافی و تبدیل مزایا و معایب به فهرست پایتون.

**متغیرها و توابع:** MAP نگاشت نام ستون‌هاست؛ در صورت تفاوت فایل آن را اصلاح کنید. clean_text متن را تمیز می‌کند، parse_list آرایه‌های متنی را می‌خواند و df داده نهایی است.

**ارتباط با روش RecMind:** مرورهای پاک‌شده، حافظه‌ای از تجربه‌های پیشین کاربر و شواهد لازم برای استنباط ترجیح می‌سازند.

In [ ]:
# نگاشت ستون‌های دیتاست keifiat.xlsx
MAP = {
    "user": "user_id",
    "item": "product_id",
    "category": "title_en",
    "verdict": "recommend",
    "comment": "comment",
    "advantages": "advantages",
    "disadvantages": "disadvantages",
    "time": None
}

def clean_text(v):
    if pd.isna(v):
        return ""
    # حذف کاراکترهای مخفی اکسل مانند _x000D_ و شکست‌های خط
    return re.sub(r"\s+", " ", str(v).replace("_x000D_", " ").replace("\r", " ").replace("\n", " ")).strip()

def parse_list(v):
    s = clean_text(v)
    if not s or s.lower() in {r"\n", "nan", "none", "[]", ""}:
        return []
    try:
        x = json.loads(s)
        if isinstance(x, list):
            return [clean_text(y) for y in x if clean_text(y)]
        if isinstance(x, str):
            s = x
    except Exception:
        pass
    # جداسازی با ویرگول فارسی، ویرگول انگلیسی، پایپ و سمی‌کالن
    return [x.strip(" []\"'") for x in re.split(r"[,;|\u060C]+", s) if x.strip(" []\"'")]

# فیلتر سطرهایی که بازخورد صریح (recommended یا not_recommended) دارند
df = df_raw[df_raw[MAP["verdict"]].astype(str).str.lower().str.strip().isin(["recommended", "not_recommended"])].copy()

df["_u"] = df[MAP["user"]].astype(str)
df["_i"] = df[MAP["item"]].astype(str)
df["_cat"] = df[MAP["category"]].map(clean_text)
df["_v"] = df[MAP["verdict"]].astype(str).str.lower().str.strip()
df["_text"] = df[MAP["comment"]].map(clean_text)
df["_pros"] = df[MAP["advantages"]].map(parse_list)
df["_cons"] = df[MAP["disadvantages"]].map(parse_list)
df = df.reset_index(drop=True)

print(f"✅ تعداد سطرهای معتبر و برچسب‌دار: {len(df):,}")
print("نمونه سطر فیلتر شده:")
display(df[["_u", "_i", "_v", "_text", "_pros", "_cons"]].head(2))


## بخش ۴ — فیلتر کاربران و ساخت بنچمارک

**هدف این سلول:** انتخاب کاربران دارای دست‌کم چهار مرور، گزینش آخرین کالای توصیه‌شده به‌عنوان هدف و ساخت چهار نامزد منفی از همان دسته title_en که کاربر ندیده است. نمونه‌گیری تا ۴۰ کاربر و با seed ثابت، طبقه‌بندی‌شده برحسب دسته انجام می‌شود.

**متغیرها و توابع:** build تاریخچه و نامزدها را می‌سازد؛ benchmark مجموعه نمونه‌هاست و در test_benchmark.json ذخیره می‌شود.

**ارتباط با روش RecMind:** سه مرور قبلی حافظه عامل‌اند و عامل باید هدف مثبت را میان پنج نامزد رتبه‌بندی کند.

In [ ]:
# فیلتر کاربرانی با تعامل کافی (حداقل ۴ تعامل برای استخراج تاریخچه + هدف)
users = df._u.value_counts()
users = users[users >= 4].index.tolist()

def build(u):
    rows = df[df._u == u]
    pos = rows[rows._v == "recommended"]
    if pos.empty:
        return None
    
    t = pos.iloc[-1]
    hist = rows.loc[:t.name].iloc[:-1].tail(3)
    
    # استخر اقلام منفی از همان دسته‌بندی که کاربر با آن‌ها تعاملی نداشته است
    pool = df[(df._cat == t._cat) & (~df._i.isin(set(rows._i)))].drop_duplicates("_i")
    if len(pool) < 4:
        return None
    
    seed_u = SEED + sum(map(ord, u))
    neg = pool.sample(4, random_state=seed_u)
    
    cs = [{
        "item_id": t._i,
        "title": f"Product {t._i} ({t._cat})",
        "comment": t["_text"],
        "advantages": t["_pros"],
        "disadvantages": t["_cons"]
    }]
    
    for _, r in neg.iterrows():
        cs.append({
            "item_id": r._i,
            "title": f"Product {r._i} ({r._cat})",
            "comment": r["_text"],
            "advantages": r["_pros"],
            "disadvantages": r["_cons"]
        })
    
    # بر زدن کاندیداها به صورت قطعی و نام‌گذاری کاندیداها از item_1 تا item_5
    random.Random(seed_u).shuffle(cs)
    for i, c in enumerate(cs, 1):
        c["candidate_id"] = f"item_{i}"
        
    return {
        "user_id": u,
        "category": t._cat,
        "history": [
            {
                "verdict": r._v,
                "comment": r["_text"],
                "advantages": r["_pros"],
                "disadvantages": r["_cons"]
            } for _, r in hist.iterrows()
        ],
        "candidates": cs,
        "ground_truth": next(c["candidate_id"] for c in cs if c["item_id"] == t._i)
    }

# ساخت تمام نمونه‌های معتبر
all_samples = [x for u in users if (x := build(u))]

# نمونه‌گیری متوازن دسته‌ای (Stratified) برای ساخت ۴۰ نمونه بنچمارک نهایی
rng = random.Random(SEED)
strata = {}
for sample in all_samples:
    strata.setdefault(sample["category"], []).append(sample)

for group in strata.values():
    rng.shuffle(group)

benchmark = []
while len(benchmark) < min(40, len(all_samples)) and any(strata.values()):
    for category in sorted(strata):
        if strata[category] and len(benchmark) < 40:
            benchmark.append(strata[category].pop())

# ذخیره بنچمارک نهایی در دیسک
benchmark_path = PROJECT_DIR / "test_benchmark.json"
benchmark_path.write_text(json.dumps(benchmark, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✅ کل نمونه‌های واجد شرایط: {len(all_samples):,}")
print(f"✅ بنچمارک استاندارد ۴۰ تایی با موفقیت ساخته و ذخیره شد در: {benchmark_path}")
print(f"نمونه اول Ground Truth: {benchmark[0]['ground_truth']}")


## بخش ۵ — پرامپت و رابط عامل

**هدف این سلول:** تعریف دستور عامل، قالب خروجی JSON و اجرای درخواست با retry و مکث افزایشی در خطا.

**متغیرها و توابع:** SYSTEM_PROMPT دستور عامل، OUTPUT_SCHEMA توصیف ساختار خروجی و run_recmind_agent فراخوانی و اعتبارسنجی رتبه‌بندی را انجام می‌دهد. مدل پیش‌فرض gpt-4o-mini است و می‌توان آن را به gpt-4o تغییر داد.

**ارتباط با روش RecMind:** عامل ابتدا ترجیح‌های پایدار را از تاریخچه خلاصه می‌کند و سپس بر پایه شواهد نامزدها را رتبه‌بندی می‌کند.

In [ ]:
import time

SYSTEM_PROMPT = """You are RecMind, an expert recommendation reasoning agent.
Task:
1. Infer the user's implicit preferences, quality sensitivities, and feature priorities strictly from their interaction history.
2. Evaluate and rank the 5 candidate items based ONLY on supplied evidence (titles, comments, advantages, disadvantages).
3. Return a valid JSON object with EXACTLY these three keys:
   - "ranking": a list of exactly 5 candidate IDs (e.g. ["item_1", "item_2", "item_3", "item_4", "item_5"]) ordered from best to worst recommendation.
   - "user_preference_summary": a concise summary of the user's inferred preferences.
   - "explanation": reasoning behind the top recommendation.
"""

def run_recmind_agent(sample, client, model="gpt-4o-mini", max_retries=3):
    expected_ids = {c["candidate_id"] for c in sample["candidates"]}
    
    # محتوای ورودی به ایجنت به صورت JSON
    input_payload = {
        "user_id": sample["user_id"],
        "category": sample["category"],
        "history": sample["history"],
        "candidates": sample["candidates"]
    }
    
    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": json.dumps(input_payload, ensure_ascii=False)}
                ],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            
            raw_text = response.choices[0].message.content
            out = json.loads(raw_text)
            
            # اعتبارسنجی کلیدها و ۵ کاندیدا
            ranking = out.get("ranking", [])
            if not isinstance(ranking, list) or len(ranking) != 5 or set(ranking) != expected_ids:
                raise ValueError(f"Invalid ranking format or IDs mismatch. Expected: {expected_ids}, Got: {ranking}")
            
            if "user_preference_summary" not in out or "explanation" not in out:
                raise ValueError("Missing required explanation keys in JSON response.")
                
            return out
            
        except Exception as e:
            last_error = e
            if attempt + 1 == max_retries:
                raise RuntimeError(f"Agent failed after {max_retries} attempts: {e}") from last_error
            sleep_time = 2 ** attempt
            print(f"⚠️ تلاش {attempt + 1} با خطا مواجه شد ({e}). تلاش مجدد پس از {sleep_time} ثانیه...")
            time.sleep(sleep_time)

print("✅ ایجنت RecMind با پرامپت استاندارد و اعتبارسنجی مقاوم تعریف شد.")


## بخش ۶ — ارزیابی دسته‌ای

**هدف این سلول:** اجرای عامل برای تمام نمونه‌ها، نگهداری پاسخ و خطای هر کاربر و ذخیره جزئیات در فایل JSON.

**متغیرها و توابع:** results شامل رتبه‌بندی، خلاصه ترجیح، توضیح و خطا است؛ tqdm نوار پیشرفت را نمایش می‌دهد.

**ارتباط با روش RecMind:** خروجی یکسان و قابل‌بازبینی برای محاسبه معیارهای توصیه تولید می‌شود.

In [ ]:
results_file = PROJECT_DIR / "agent_evaluation_results.json"

# بارگذاری نتایج قبلی در صورت وجود (امکان Resume)
results = []
done_users = set()

if results_file.exists():
    try:
        existing_data = json.loads(results_file.read_text(encoding="utf-8"))
        if isinstance(existing_data, list):
            results = existing_data
            done_users = {r["user_id"] for r in results if r.get("selected_recommendation") is not None}
            print(f"🔄 بازخوانی نتایج قبلی: {len(done_users)} نمونه قبلاً با موفقیت ارزیابی شده‌اند.")
    except Exception as e:
        print(f"⚠️ امکان خواندن نتایج قبلی نبود ({e}). اجرای ارزیابی از ابتدا آغاز می‌شود.")
        results = []

eval_model = MODEL if "MODEL" in globals() else "gpt-4o-mini"
print(f"🚀 شروع ارزیابی بنچمارک با مدل: {eval_model}")

for s in tqdm(benchmark, desc="Evaluating Agents"):
    uid = s["user_id"]
    if uid in done_users:
        continue  # قبلاً پردازش شده است
        
    r = {
        "user_id": uid,
        "category": s.get("category", ""),
        "ground_truth": s["ground_truth"],
        "candidates": s["candidates"]
    }
    
    try:
        o = run_recmind_agent(s, client, model=eval_model)
        r.update(o)
        r["selected_recommendation"] = o["ranking"][0]
        r["error"] = None
    except Exception as e:
        print(f"\n❌ خطا در پردازش کاربر {uid}: {e}")
        r.update({
            "ranking": [],
            "user_preference_summary": "",
            "explanation": "",
            "error": str(e),
            "selected_recommendation": None
        })
        
    # به‌روزرسانی لیست و ذخیره‌سازی اتمیک بعد از هر درخواست
    results = [item for item in results if item["user_id"] != uid]
    results.append(r)
    
    # ذخیره در فایل موقت و انتقال امن (Atomic Write)
    temp_file = results_file.with_suffix(".tmp")
    temp_file.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    temp_file.replace(results_file)

successful_runs = sum(1 for x in results if x.get("selected_recommendation") is not None)
print(f"\n🎯 ارزیابی بنچمارک به پایان رسید!")
print(f"📊 کل موارد ارزیابی‌شده: {len(results)} | موفق: {successful_runs}/{len(benchmark)}")


## بخش ۷ — معیارها و مقایسه با خط مبنا

**هدف این سلول:** محاسبه HR@1، HR@3، HR@5 و NDCG@5 و نمایش جدول مقایسه با انتخاب تصادفی.

**متغیرها و توابع:** ranking_metrics معیارهای مدل را محاسبه می‌کند؛ metrics و random_baseline در comparison قرار می‌گیرند. با یک هدف مثبت میان پنج نامزد، امید NDCG@5 تصادفی استاندارد حدود ۰٫۵۹۰ است.

**ارتباط با روش RecMind:** مقایسه با خط مبنا نشان می‌دهد رتبه‌بندی عامل تا چه حد بهتر از انتخاب تصادفی عمل می‌کند.

In [ ]:
def ranking_metrics(rows):
    # فیلتر نمونه‌های معتبری که بدون خطا رتبه‌بندی شده‌اند
    valid_rows = [r for r in rows if not r.get("error") and r.get("ranking") and r["ground_truth"] in r["ranking"]]
    
    if not valid_rows:
        return {
            "Valid_Samples": 0,
            "HR@1": np.nan,
            "HR@3": np.nan,
            "HR@5": np.nan,
            "MRR": np.nan,
            "NDCG@5": np.nan
        }
    
    ranks = [r["ranking"].index(r["ground_truth"]) + 1 for r in valid_rows]
    ranks_arr = np.array(ranks)
    
    hr1 = np.mean(ranks_arr <= 1)
    hr3 = np.mean(ranks_arr <= 3)
    hr5 = np.mean(ranks_arr <= 5)
    mrr = np.mean(1.0 / ranks_arr)
    # با توجه به اینکه IDCG=1 است:
    ndcg5 = np.mean([1.0 / math.log2(k + 1) if k <= 5 else 0.0 for k in ranks])
    
    return {
        "Valid_Samples": len(valid_rows),
        "HR@1": hr1,
        "HR@3": hr3,
        "HR@5": hr5,
        "MRR": mrr,
        "NDCG@5": ndcg5
    }

# محاسبه متریک‌های ایجنت RecMind
metrics = ranking_metrics(results)

# بیس‌لاین رندوم تئوریک برای ارزیابی رتبه‌بندی ۵ تایی
rand_ranks = list(range(1, 6))
random_baseline = {
    "Valid_Samples": metrics["Valid_Samples"],
    "HR@1": 1.0 / 5,                                       # 0.2000
    "HR@3": 3.0 / 5,                                       # 0.6000
    "HR@5": 5.0 / 5,                                       # 1.0000
    "MRR": np.mean([1.0 / k for k in rand_ranks]),         # ~0.4567
    "NDCG@5": np.mean([1.0 / math.log2(k + 1) for k in rand_ranks]) # ~0.5516
}

# ساخت جدول مقایسه‌ای نهایی
comparison_df = pd.DataFrame([metrics, random_baseline], index=["RecMind (Agent)", "Random Baseline"])

# محاسبه درصد بهبود (Lift %) نسبت به رندوم
if metrics["Valid_Samples"] > 0:
    lift = {}
    for col in ["HR@1", "HR@3", "MRR", "NDCG@5"]:
        base_val = random_baseline[col]
        agent_val = metrics[col]
        lift[col] = f"{((agent_val - base_val) / base_val) * 100:+.2f}%"
    lift["Valid_Samples"] = "-"
    lift["HR@5"] = "0.00%"
    comparison_df.loc["Improvement (Lift)"] = lift

print("📊 جدول نتایج نهایی بنچمارک سیستم توصیه‌گر عامل‌محور:")
display(comparison_df)


## بخش ۸ — تحلیل کیفی و مطالعه موردی

**هدف این سلول:** نمایش تاریخچه و مزایا و معایب دو کاربر، پنج نامزد، خلاصه ترجیح و توضیح عامل و صحت انتخاب هدف.

**متغیرها و توابع:** show_case جزئیات هر نمونه را چاپ می‌کند؛ by_user و samples برای دسترسی به نتایج و بنچمارک استفاده می‌شوند.

**ارتباط با روش RecMind:** بررسی نمونه‌ای نشان می‌دهد عامل چگونه از حافظه کاربر برای استدلال درباره ویژگی‌های مطلوب استفاده کرده است.

In [ ]:
by_user = {r["user_id"]: r for r in results}
samples = {s["user_id"]: s for s in benchmark}

def show_case(u):
    s = samples[u]
    r = by_user[u]
    gt = s["ground_truth"]
    ranking = r.get("ranking", [])
    agent_pick = ranking[0] if ranking else None
    is_hit_at_1 = (agent_pick == gt)
    rank_of_gt = (ranking.index(gt) + 1) if (ranking and gt in ranking) else "N/A"
    
    print("=" * 90)
    print(f"👤 کاربر: {u} | 🏷️ دسته‌بندی: {s['category']}")
    print(f"🎯 کاندیدای هدف (Ground Truth): {gt} | 🏆 انتخاب ایجنت (Rank 1): {agent_pick} | رتبه آیتم هدف: {rank_of_gt}")
    print(f"نتیجه HR@1: {'✅ موفق (Hit)' if is_hit_at_1 else '❌ ناموفق (Miss)'}")
    print("-" * 90)
    
    print("📜 تاریخچه تعاملات کاربر (User History):")
    df_hist = pd.DataFrame(s["history"])
    display(df_hist)
    
    print("📦 گزینه‌های کاندیدا (Candidates) و رتبه‌بندی مدل:")
    df_cand = pd.DataFrame(s["candidates"])
    # اضافه کردن رتبه مدل به جدول کاندیداها برای بررسی سریع
    if ranking:
        rank_map = {cid: idx + 1 for idx, cid in enumerate(ranking)}
        df_cand["agent_rank"] = df_cand["candidate_id"].map(rank_map)
        df_cand["is_ground_truth"] = df_cand["candidate_id"] == gt
        df_cand = df_cand.sort_values("agent_rank").reset_index(drop=True)
    display(df_cand)
    
    print("🧠 استنباط ایجنت از علایق کاربر (User Preference Summary):")
    print(f"   {r.get('user_preference_summary')}\n")
    
    print("💡 استدلال و چرایی انتخاب رتبه اول (Explanation):")
    print(f"   {r.get('explanation')}\n")
    print("=" * 90 + "\n")

# نمایش ۲ نمونه اول بنچمارک جهت تحلیل کیفی
for u in list(samples)[:2]:
    show_case(u)
